In [438]:
import pandas as pd
import openpyxl
import os
import glob, re

In [439]:
def read_excel(files, sheet):
    dfs = []
    for file in files:
        data = pd.read_excel(file, sheet_name=sheet, index_col=0)
        name = os.path.splitext(os.path.basename(file))[0]
        data['model'] = name
        match = re.search(r'lr[\d.]+_(\d+)_', name)
        if match:
            data['epochs'] = int(match.group(1))
        match_model = re.search(r'model_.*?_e_(.*?)_lr', name)
        if match_model:
            data['id'] = match_model.group(1)
        dfs.append(data)
    df = pd.concat(dfs,ignore_index=True)
    return df

In [440]:
def arrange_data(df, error):
    df = df.pivot(index='model', columns='error', values=error)
    return df

In [441]:
def min_vals(df, error):
    df1 = df.apply(lambda x: pd.Series({'model':x.idxmin(), error:x.min()})).T
    return df1

In [442]:
path = "model_Fe_Si_B_260311/**/test_res/*_test.xlsx"

In [443]:
files = glob.glob(path, recursive=True)

In [444]:
df = read_excel(files, 'errors')
df = df[['model', 'id', 'epochs']+[c for c in df.columns if c not in ['model', 'id', 'epochs']]]
df

,model,id,epochs,error,rmse,mae,r2
0,model_rnd_e_scmace_lr0.0001_100_10_test,scmace,100,test_energy,210.180375,124.774518,0.760373
1,model_rnd_e_scmace_lr0.0001_100_10_test,scmace,100,train_energy,212.670122,104.387671,0.779457
2,model_rnd_e_scmace_lr0.0001_100_10_test,scmace,100,test_force,0.473279,0.266766,0.971340
3,model_rnd_e_scmace_lr0.0001_100_10_test,scmace,100,train_force,0.350139,0.264250,0.984159
4,model_rnd_e_matpes_lr0.0001_80_10_test,matpes,80,test_energy,43.019987,26.794243,0.989961
5,model_rnd_e_matpes_lr0.0001_80_10_test,matpes,80,train_energy,100.808691,27.309964,0.950446
6,model_rnd_e_matpes_lr0.0001_80_10_test,matpes,80,test_force,0.159534,0.120127,0.996744
7,model_rnd_e_matpes_lr0.0001_80_10_test,matpes,80,train_force,0.139589,0.107203,0.997482
8,model_rnd_e_matpes_lr0.0001_40_10_test,matpes,40,test_energy,49.126072,28.338427,0.986909
9,model_rnd_e_matpes_lr0.0001_40_10_test,matpes,40,train_energy,103.295626,27.930106,0.947971


In [445]:
#df.loc[df['model']=='model_rnd_e_matpes_lr0.0001_40_10_test']

In [446]:
rmse = arrange_data(df, 'rmse')
mae = arrange_data(df, 'mae')
rmse

error,test_energy,test_force,train_energy,train_force
model,,,,
model_rnd_e_matpes_lr0.0001_40_10_test,49.126072,0.162873,103.295626,0.152998
model_rnd_e_matpes_lr0.0001_60_10_test,46.366162,0.160293,102.324352,0.146102
model_rnd_e_matpes_lr0.0001_80_10_test,43.019987,0.159534,100.808691,0.139589
model_rnd_e_scmace_lr0.0001_100_10_test,210.180375,0.473279,212.670122,0.350139
model_rnd_e_scmace_lr0.0001_80_10_test,230.282754,0.520437,232.658891,0.385412


In [447]:
min_vals(rmse, 'rsme')

,model,rsme
error,,
test_energy,model_rnd_e_matpes_lr0.0001_80_10_test,43.019987
test_force,model_rnd_e_matpes_lr0.0001_80_10_test,0.159534
train_energy,model_rnd_e_matpes_lr0.0001_80_10_test,100.808691
train_force,model_rnd_e_matpes_lr0.0001_80_10_test,0.139589


In [448]:
min_vals(mae, 'mae')

,model,mae
error,,
test_energy,model_rnd_e_matpes_lr0.0001_80_10_test,26.794243
test_force,model_rnd_e_matpes_lr0.0001_80_10_test,0.120127
train_energy,model_rnd_e_matpes_lr0.0001_80_10_test,27.309964
train_force,model_rnd_e_matpes_lr0.0001_80_10_test,0.107203


In [449]:
#df.loc[df['id']=='matpes']

In [450]:
#df.query('id == "matpes" & epochs in [80, 40]')

In [451]:
#fine_tuning = df.query('id == "matpes"')
#min_vals(fine_tuning, 'mae')

Reading the config errors

In [452]:
df_config =read_excel(files, 'config_errors')
df_config = df_config[['model', 'id', 'epochs']+[c for c in df_config.columns if c not in ['model', 'id', 'epochs']]]
df_config

,model,id,epochs,error,n_configs,rmse,mae
0,model_rnd_e_scmace_lr0.0001_100_10_test,scmace,100,test_energy,48,393.680934,392.463504
1,model_rnd_e_scmace_lr0.0001_100_10_test,scmace,100,test_energy,12,136.318493,134.133466
2,model_rnd_e_scmace_lr0.0001_100_10_test,scmace,100,test_energy,11,72.981237,66.959629
3,model_rnd_e_scmace_lr0.0001_100_10_test,scmace,100,test_energy,133,15.474716,12.602748
4,model_rnd_e_scmace_lr0.0001_100_10_test,scmace,100,test_energy,8,448.967178,448.953319
5,model_rnd_e_scmace_lr0.0001_100_10_test,scmace,100,train_energy,132,388.063913,386.640402
6,model_rnd_e_scmace_lr0.0001_100_10_test,scmace,100,train_energy,39,139.183731,137.138739
7,model_rnd_e_scmace_lr0.0001_100_10_test,scmace,100,train_energy,40,66.693207,56.930606
8,model_rnd_e_scmace_lr0.0001_100_10_test,scmace,100,train_energy,590,15.470571,12.430298
9,model_rnd_e_scmace_lr0.0001_100_10_test,scmace,100,train_energy,44,626.410301,504.807675
